In [1]:
import numpy as np
import pandas as pd

tc = 0.001
H = 25

In [2]:
# --- Load saved step 2 outputs ---
preds = pd.read_csv("outputs/step2_single_output_predictions.csv", index_col=0, parse_dates=True)
acts  = pd.read_csv("outputs/step2_single_output_actuals.csv", index_col=0, parse_dates=True)

# Columns in file are like "pred_Equity (SPY)"; clean them
preds.columns = [c.replace("pred_", "") for c in preds.columns]
acts.columns  = [c.replace("actual_", "") for c in acts.columns]

# Align
common_idx = preds.index.intersection(acts.index)
preds = preds.loc[common_idx].sort_index()
acts  = acts.loc[common_idx].sort_index()

assets = list(preds.columns)

# Rebalance schedule
rebalance_idx = preds.index[::H]

def build_long_short_weights(pred_row, n_long=2, n_short=2):
    ranked = pred_row.sort_values(ascending=False)
    longs = ranked.index[:n_long]
    shorts = ranked.index[-n_short:]
    w = pd.Series(0.0, index=pred_row.index)
    w.loc[longs] =  1.0 / n_long
    w.loc[shorts] = -1.0 / n_short
    return w

weights = pd.DataFrame(index=rebalance_idx, columns=assets, dtype=float)
for dt in rebalance_idx:
    weights.loc[dt] = build_long_short_weights(preds.loc[dt], 2, 2)

# Turnover and costs
turnover = 0.5 * weights.diff().abs().sum(axis=1)
turnover.iloc[0] = 0.5 * weights.iloc[0].abs().sum()

# Backtest on realized 25-day forward returns (already in acts)
strategy_gross = []
strategy_net = []
for dt in rebalance_idx:
    w = weights.loc[dt].astype(float)
    realized = acts.loc[dt].astype(float)
    r_g = float((w * realized).sum())
    cost = tc * float(turnover.loc[dt])
    r_n = r_g - cost
    strategy_gross.append(r_g)
    strategy_net.append(r_n)

strategy_gross = pd.Series(strategy_gross, index=rebalance_idx, name="strategy_gross_25d_logret")
strategy_net   = pd.Series(strategy_net,   index=rebalance_idx, name="strategy_net_25d_logret")

In [3]:
def perf_stats(r):
    r = pd.Series(r).dropna()
    ann = 252/25  # 25 trading days per rebalance
    ann_mean_log = r.mean() * ann
    ann_vol = r.std(ddof=0) * np.sqrt(ann)
    sharpe = ann_mean_log / ann_vol if ann_vol > 0 else np.nan
    eq = np.exp(r.cumsum())
    peak = eq.cummax()
    dd = (eq/peak - 1.0).min()
    return pd.Series({
        "ann_mean_log": ann_mean_log,
        "ann_vol": ann_vol,
        "sharpe_(rf=0)": sharpe,
        "max_drawdown": dd,
        "final_equity": float(eq.iloc[-1])
    })

# benchmark series must be computed on SAME rebalance dates:
# easiest: equal-weight of realized forward returns
bench_25d = acts.loc[rebalance_idx].mean(axis=1).rename("bench_25d_logret")

stats_step2 = pd.DataFrame({
    "Strategy (Gross)": perf_stats(strategy_gross),
    "Strategy (Net)": perf_stats(strategy_net),
    "Benchmark (EW B/H)": perf_stats(bench_25d)
})

table10 = stats_step2.loc[["ann_mean_log","ann_vol","sharpe_(rf=0)","max_drawdown","final_equity"]].T
table10["max_drawdown"] = (table10["max_drawdown"]*100).round(2).astype(str) + "%"
table10

,ann_mean_log,ann_vol,sharpe_(rf=0),max_drawdown,final_equity
Strategy (Gross),-0.066096,0.174102,-0.379637,-27.38%,0.744480
Strategy (Net),-0.075280,0.174279,-0.431948,-28.64%,0.714573
Benchmark (EW B/H),0.046409,0.080793,0.574415,-11.69%,1.230208


In [4]:
table17_step2 = pd.DataFrame({
    "Metric": [
        "Average Turnover per Rebalance",
        "Total Transaction Cost (sum(tc*turnover))",
        "Transaction Cost Rate (tc)",
        "Number of Rebalances"
    ],
    "Value": [
        float(turnover.mean()),
        float((tc * turnover).sum()),
        float(tc),
        int(len(turnover))
    ]
})
table17_step2

,Metric,Value
0,Average Turnover per Rebalance,0.911111
1,Total Transaction Cost (sum(tc*turnover)),0.041000
2,Transaction Cost Rate (tc),0.001000
3,Number of Rebalances,45.000000


In [5]:
preds3 = pd.read_csv("outputs/step3_multi_output_predictions_REG.csv", index_col=0, parse_dates=True)
acts3  = pd.read_csv("outputs/step3_multi_output_actuals_REG.csv", index_col=0, parse_dates=True)

common_idx3 = preds3.index.intersection(acts3.index)
preds3 = preds3.loc[common_idx3].sort_index()
acts3  = acts3.loc[common_idx3].sort_index()

assets3 = list(preds3.columns)
rebalance_idx3 = preds3.index[::H]

weights_multi = pd.DataFrame(index=rebalance_idx3, columns=assets3, dtype=float)
for dt in rebalance_idx3:
    weights_multi.loc[dt] = build_long_short_weights(preds3.loc[dt], 2, 2)

turnover_multi = 0.5 * weights_multi.diff().abs().sum(axis=1)
turnover_multi.iloc[0] = 0.5 * weights_multi.iloc[0].abs().sum()

multi_net = []
for dt in rebalance_idx3:
    w = weights_multi.loc[dt].astype(float)
    realized = acts3.loc[dt].astype(float)
    r_g = float((w * realized).sum())
    cost = tc * float(turnover_multi.loc[dt])
    multi_net.append(r_g - cost)

multi_net = pd.Series(multi_net, index=rebalance_idx3, name="multi_net_25d_logret")

bench3 = acts3.loc[rebalance_idx3].mean(axis=1).rename("bench_25d_logret")

In [6]:
# IMPORTANT: compare on SAME rebalance index intersection
common_reb = strategy_net.index.intersection(multi_net.index).intersection(bench3.index)

stats_step3 = pd.DataFrame({
    "Multi-output (Net)": perf_stats(multi_net.loc[common_reb]),
    "Single-output (Net)": perf_stats(strategy_net.loc[common_reb]),
    "Benchmark (EW B/H)": perf_stats(bench3.loc[common_reb])
})

table15 = stats_step3.loc[["ann_mean_log","ann_vol","sharpe_(rf=0)","max_drawdown","final_equity"]].T
table15["max_drawdown"] = (table15["max_drawdown"]*100).round(2).astype(str) + "%"
table15

,ann_mean_log,ann_vol,sharpe_(rf=0),max_drawdown,final_equity
Multi-output (Net),0.206177,0.195214,1.056160,-18.09%,2.510375
Single-output (Net),-0.075280,0.174279,-0.431948,-28.64%,0.714573
Benchmark (EW B/H),0.046409,0.080793,0.574415,-11.69%,1.230208
